# Agent chạy model thật trên Colab (Ollama + qwen3:4b)

Notebook này cài Ollama, tải model `qwen3:4b` (khoảng 2,5 GB), rồi cho agent của repo làm 5 nhiệm vụ mẫu bằng 2 công cụ:
- **calculator**: tính biểu thức số học;
- **terminal** (TerminalTool): chạy lệnh trong danh sách cho phép như `ls`, `cat`, chỉ trong thư mục làm việc riêng của từng nhiệm vụ.

Agent gọi model qua adapter kiểu OpenAI của repo (`http://localhost:11434/v1`). Mỗi nhiệm vụ in trace (kế hoạch, công cụ đã gọi, kết quả), cuối cùng in tỉ lệ thành công.

**Không cần token Hugging Face.** Nên chọn GPU T4 (menu Runtime → Change runtime type); chạy trên CPU cũng được nhưng rất chậm.

**Cách chạy:** menu Runtime → Run all. Thời gian ước tính: cài đặt và tải model khoảng 5 phút, 5 nhiệm vụ khoảng 5–15 phút (qwen3 suy nghĩ trước khi trả lời). Con số này chưa đo trên Colab thật.

In [ ]:
# Bước 1: kiểm tra GPU. Ollama chạy được trên CPU nhưng chậm hơn nhiều.
# Nên chọn T4: menu Runtime → Change runtime type → T4 GPU → Save, rồi Run all lại.
import shutil
import subprocess

if shutil.which("nvidia-smi"):
    print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True).stdout)
else:
    print("Chưa có GPU: vẫn chạy được trên CPU nhưng rất chậm. Nên chọn T4 GPU rồi Run all lại.")

In [ ]:
# Bước 2: tải code của repo. Phần agent chỉ dùng thư viện có sẵn của Python, không cần cài thêm gói pip nào.
# Từ đây, lệnh nào lỗi thì ô báo đỏ và Run all dừng ở ô đó.
import os

if not os.path.isdir("/content/Huyen/local_ai"):
    !git clone --depth 1 https://github.com/hytmk2912/Huyen.git /content/Huyen
%cd /content/Huyen
if not os.path.isdir("local_ai"):
    raise RuntimeError("Tải code thất bại: chưa có thư mục /content/Huyen/local_ai. Kiểm tra mạng rồi chạy lại ô này.")
from local_ai.colab import run  # chạy lệnh: lệnh lỗi thì ô báo đỏ và Run all dừng ở đúng chỗ lỗi

run("git pull --ff-only", "Bước 2 (cập nhật code)")

In [ ]:
# Bước 3: cài Ollama bản đã ghim (0.34.4). zstd dùng để giải nén bản cài, pciutils giúp Ollama nhận ra GPU.
run("apt-get -qq install -y zstd pciutils", "Bước 3 (cài zstd)", quiet=True)
run("curl -fsSL -o /tmp/ollama_install.sh https://ollama.com/install.sh", "Bước 3 (tải bản cài Ollama)")
run(["sh", "/tmp/ollama_install.sh"], "Bước 3 (cài Ollama)", env={"OLLAMA_VERSION": "0.34.4"})
run("ollama --version", "Bước 3 (kiểm tra Ollama)")

In [ ]:
# Bước 4: chạy máy chủ Ollama ở nền (log ghi vào /content/ollama.log) và đợi tới khi nó trả lời.
# Chạy lại notebook khi Ollama đang chạy thì không mở thêm máy chủ mới.
import subprocess
import time
import urllib.request


def ollama_ready():
    try:
        urllib.request.urlopen("http://localhost:11434/api/version", timeout=2)
        return True
    except OSError:
        return False


if not ollama_ready():
    subprocess.Popen(["ollama", "serve"], stdout=open("/content/ollama.log", "w"), stderr=subprocess.STDOUT)
for _ in range(60):
    if ollama_ready():
        print("Ollama đã chạy.")
        break
    time.sleep(1)
else:
    raise RuntimeError("Ollama chưa chạy sau 60 giây; hãy xem file /content/ollama.log")

In [ ]:
# Bước 5: tải model qwen3:4b (khoảng 2,5 GB). Chạy lại thì Ollama không tải lại.
run("ollama pull qwen3:4b", "Bước 5 (tải model)")

In [ ]:
# Bước 6: kiểm tra cấu hình, chưa gọi model: 5 nhiệm vụ, công cụ mỗi nhiệm vụ cần, địa chỉ máy chủ Ollama.
run("python -m local_ai.agents.tasks --model ollama-colab --dry-run", "Bước 6 (kiểm tra cấu hình)")

In [ ]:
# Bước 7: cho agent làm 5 nhiệm vụ mẫu. Mỗi nhiệm vụ in trace (kế hoạch, công cụ đã gọi, kết quả); dòng cuối là tỉ lệ thành công.
# TerminalTool chỉ được bật trong thư mục làm việc riêng của từng nhiệm vụ (.runs/agent_tasks/<nhiệm vụ>/workspace).
run("python -m local_ai.agents.tasks --model ollama-colab --output .runs/agent_tasks/report.json", "Bước 7 (chạy agent)")

## Kết quả nằm ở đâu
- **Trace và tỉ lệ thành công:** in ngay dưới Bước 7. Chi tiết (JSON) ở `.runs/agent_tasks/report.json`; file này mất khi Colab tắt, nên hãy chụp màn hình.
- Một nhiệm vụ chỉ tính là **đạt** khi câu trả lời đúng **và** agent đã thật sự gọi công cụ cần dùng (không tính trường hợp model tự đoán).
- Model nhỏ đôi khi trả JSON sai dạng hoặc quên gọi công cụ. Khi đó nhiệm vụ không đạt; đây là điều notebook muốn đo.

## Lỗi hay gặp
- **Ô báo đỏ "Bước N lỗi (mã thoát ...)":** Run all dừng ở ô đó; đọc thông báo ngay phía trên dòng đỏ, sửa xong thì chạy lại từ ô đó.
- **"Không kết nối được server model":** Ollama chưa chạy; chạy lại Bước 4 (xem `/content/ollama.log`).
- **"không trả lời trong 300 giây":** đang chạy trên CPU hoặc máy quá tải; chọn T4 GPU rồi Run all lại.
- **Bước 3 báo cần zstd:** chạy lại Bước 3 (lệnh apt-get cài zstd nằm ngay đầu ô).
- **`ollama pull` lỗi mạng:** chạy lại Bước 5; phần đã tải được giữ lại.